# Preprocessing the Three Datasets

Covers the preprocessing work for our
Gini-vs-Entropy interpretability project (see
`project-work/aaron/research/datasets.md` and `project-work/skye/idea-proposal.md`).

Per James's guidance (2026-09-28 meeting), we pull the real datasets from the
**UCI Machine Learning Repository** via its official `ucimlrepo` package
rather than using the pre-cleaned toy copies bundled with `sklearn.datasets`,
so the preprocessing work (missing values, encoding, target construction) is
ours to do and document.

Datasets (see `project-work/aaron/research/datasets.md` for why these three):

1. [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) (id 17)
2. [Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease) (id 45)
3. [Heart Failure Clinical Records](https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records) (id 519)

Companion notebook: `02-gini-vs-entropy-explainability.ipynb` (training +
interpretability comparison, once the data is in shape).

## 0. Setup

In [ ]:
# If running in Colab, uncomment:
# !pip install ucimlrepo -q

from pathlib import Path

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42  # matches code/config.py
TEST_SIZE = 0.2    # matches code/config.py

OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Breast Cancer Wisconsin (Diagnostic)

In [ ]:
bc = fetch_ucirepo(id=17)
print(bc.metadata.name)
print(f"Instances: {bc.metadata.num_instances}, Features: {bc.metadata.num_features}")
print(f"DOI: {bc.metadata.additional_info.get('summary', '')[:0]}")  # (kept for parity; see citation below)
print("Citation / DOI: https://doi.org/10.24432/C5DW2B")

X_bc_raw = bc.data.features.copy()
y_bc_raw = bc.data.targets.copy().iloc[:, 0]

print(f"\nFeatures shape: {X_bc_raw.shape}")
print(f"Missing values total: {X_bc_raw.isna().sum().sum()}")
print(f"\nTarget value counts:\n{y_bc_raw.value_counts()}")

**Preprocessing decisions:**
- All 30 features are already continuous numeric measurements (no categorical
  encoding needed).
- No missing values.
- Target `Diagnosis` is `M`/`B` (malignant/benign) — encode as `1`/`0`.
- No feature scaling: decision trees split on thresholds per feature
  independently, so they're invariant to monotonic rescaling. (This would
  matter for distance-based models, but not here.)

In [ ]:
def load_breast_cancer():
    ds = fetch_ucirepo(id=17)
    X = ds.data.features.copy()
    y = (ds.data.targets.copy().iloc[:, 0] == "M").astype(int)  # 1 = malignant
    return X, y


X_bc, y_bc = load_breast_cancer()
X_bc.describe().T.head()

## 2. Heart Disease

In [ ]:
hd = fetch_ucirepo(id=45)
print(hd.metadata.name)
print(f"Instances: {hd.metadata.num_instances}, Features: {hd.metadata.num_features}")
print("Citation / DOI: https://doi.org/10.24432/C52P4X")

X_hd_raw = hd.data.features.copy()
y_hd_raw = hd.data.targets.copy().iloc[:, 0]

print(f"\nFeatures shape: {X_hd_raw.shape}")
print(f"\nMissing values per column:\n{X_hd_raw.isna().sum()[X_hd_raw.isna().sum() > 0]}")
print(f"\nRaw target value counts (0 = no disease, 1-4 = increasing severity):\n{y_hd_raw.value_counts().sort_index()}")
hd.variables[["name", "role", "type", "description"]]

**Preprocessing decisions:**
- Target `num` is 0-4 (severity). Following common practice for this dataset
  (and because we want a binary classification task matching the other two
  datasets), we binarize: `0` = no disease, `1+` = disease present.
- `ca` (4 missing) and `thal` (2 missing) have a handful of missing values.
  With only 303 rows we don't want to drop them — impute `ca` with the
  median (it's a small-range count, 0-3) and `thal` with the mode (it's
  categorical).
- `cp`, `restecg`, `slope`, `thal` are **nominal** categorical features
  (no inherent order) encoded as integers in the raw data — one-hot encode
  these so the tree doesn't implicitly treat them as ordered/thresholdable.
  `sex`, `fbs`, `exang` are already binary 0/1, so no encoding needed there.
- No feature scaling, same reasoning as the breast cancer dataset.

In [ ]:
NOMINAL_COLS_HD = ["cp", "restecg", "slope", "thal"]


def load_heart_disease():
    ds = fetch_ucirepo(id=45)
    X = ds.data.features.copy()
    y = (ds.data.targets.copy().iloc[:, 0] > 0).astype(int)  # 1 = any disease

    X["ca"] = X["ca"].fillna(X["ca"].median())
    X["thal"] = X["thal"].fillna(X["thal"].mode().iloc[0])

    X = pd.get_dummies(X, columns=NOMINAL_COLS_HD, prefix=NOMINAL_COLS_HD)
    X = X.astype({c: int for c in X.columns if X[c].dtype == bool})

    return X, y


X_hd, y_hd = load_heart_disease()
print(f"Shape before encoding: {X_hd_raw.shape} -> after one-hot encoding: {X_hd.shape}")
print(f"Missing values remaining: {X_hd.isna().sum().sum()}")
X_hd.head()

## 3. Heart Failure Clinical Records

In [ ]:
hf = fetch_ucirepo(id=519)
print(hf.metadata.name)
print(f"Instances: {hf.metadata.num_instances}, Features: {hf.metadata.num_features}")
print("Citation / DOI: https://doi.org/10.24432/C5Z89R")

X_hf_raw = hf.data.features.copy()
y_hf_raw = hf.data.targets.copy().iloc[:, 0]

print(f"\nFeatures shape: {X_hf_raw.shape}")
print(f"Missing values total: {X_hf_raw.isna().sum().sum()}")
print(f"\nTarget value counts:\n{y_hf_raw.value_counts()}")

**A data-quality issue worth flagging, not silently fixing:** `time` is the
patient's **follow-up period in days**, not a clinical measurement taken at
diagnosis. In survival-style datasets like this, `time` is known in the
literature to leak information about the outcome (very short follow-up
correlates strongly with death during the study). Let's check how strong
that correlation actually is here before deciding what to do about it.

In [ ]:
corr_with_target = X_hf_raw.assign(death_event=y_hf_raw).corr(numeric_only=True)["death_event"].drop("death_event")
corr_with_target.sort_values(key=np.abs, ascending=False)

`time` is by far the strongest (negative) correlate of `death_event` — much
stronger than any actual clinical variable. Including it would likely make
the tree pick it as the dominant/only split, which tells us about *study
follow-up design* rather than about the clinical features we actually care
about for an interpretability comparison.

**Decision:** exclude `time` from the feature set by default (flag kept as
`exclude_time`, defaulting to `True`, so the group can switch it back on and
see the effect for the report's limitations section if useful).

In [ ]:
def load_heart_failure(exclude_time=True):
    ds = fetch_ucirepo(id=519)
    X = ds.data.features.copy()
    y = ds.data.targets.copy().iloc[:, 0]
    if exclude_time:
        X = X.drop(columns=["time"])
    return X, y


X_hf, y_hf = load_heart_failure()
print(f"Shape before: {X_hf_raw.shape} -> after dropping 'time': {X_hf.shape}")
X_hf.head()

## 4. Train/Test Split & Save Processed Data

Consistent 80/20 stratified split across all three datasets, same random
seed, so results are comparable. Saved to CSV so `02-gini-vs-entropy-
explainability.ipynb` (or anyone else) can load the exact same processed
data without refetching/reprocessing (loader functions above are also
copied into that notebook so it can run standalone in a fresh Colab
runtime).

In [ ]:
DATASETS = {
    "breast_cancer": (X_bc, y_bc),
    "heart_disease": (X_hd, y_hd),
    "heart_failure": (X_hf, y_hf),
}

summary_rows = []

for name, (X, y) in DATASETS.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    X_train.assign(target=y_train).to_csv(OUT_DIR / f"{name}_train.csv", index=False)
    X_test.assign(target=y_test).to_csv(OUT_DIR / f"{name}_test.csv", index=False)

    summary_rows.append({
        "dataset": name,
        "n_instances": len(X),
        "n_features": X.shape[1],
        "n_train": len(X_train),
        "n_test": len(X_test),
        "positive_rate": round(y.mean(), 3),
    })
    print(f"Saved {name}: train={X_train.shape}, test={X_test.shape}")

summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. Summary

| Dataset | Instances | Features (after encoding) | Missing values handled | Notes |
|---|---|---|---|---|
| Breast Cancer Wisconsin (Diagnostic) | 569 | 30 | none needed | all continuous |
| Heart Disease | 303 | see table above (one-hot adds columns) | `ca` (median), `thal` (mode) | 4 nominal features one-hot encoded |
| Heart Failure Clinical Records | 299 | 11 (`time` excluded) | none needed | `time` excluded — leaks outcome |
